# LeetCode 1436 — Destination City bằng Python

**Link đề:** https://leetcode.com/problems/destination-city/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String

```python
class Solution:
    def destCity(self, paths: list[list[str]]) -> str:
        ...
```

### Mục tiêu học
1. Đọc được dữ liệu dạng **list lồng list** (`paths[i] = [A, B]`) và **unpacking** `for a, b in paths`.
2. Hiểu và dùng thành thạo **`set`** (tập hợp): thêm, kiểm tra `in`, phép hiệu `-`.
3. Dùng **`dict`** để mô tả "từ thành phố này đi tiếp tới đâu" và **đi theo chuỗi**.
4. Đi qua 4 lời giải: brute force `O(n²)` → set nguồn `O(n)` → đi theo chuỗi `O(n)` → hiệu tập hợp.
5. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, luyện biến thể (tìm điểm xuất phát, in cả lộ trình).

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — tìm thành phố đích

Cho mảng `paths`, trong đó `paths[i] = [cityA_i, cityB_i]` nghĩa là có **đường đi trực tiếp** từ `cityA_i` tới `cityB_i`.
Trả về **thành phố đích**: thành phố **không có đường đi ra** tới thành phố nào khác.

Đề **đảm bảo** các đường đi tạo thành **một đường thẳng không có vòng**, nên có **đúng một** thành phố đích.

| Ví dụ | `paths` | Output | Giải thích |
|---|---|---|---|
| 1 | `[["London","New York"],["New York","Lima"],["Lima","Sao Paulo"]]` | `"Sao Paulo"` | London → New York → Lima → Sao Paulo |
| 2 | `[["B","C"],["D","B"],["C","A"]]` | `"A"` | D → B → C → A (thứ tự trong mảng bị xáo trộn!) |
| 3 | `[["A","Z"]]` | `"Z"` | chỉ một đường A → Z |

**Constraints**
- `1 <= paths.length <= 100`
- `paths[i].length == 2`
- `1 <= cityA_i.length, cityB_i.length <= 10`
- `cityA_i != cityB_i`
- Tên thành phố gồm chữ cái hoa/thường và **dấu cách** (ví dụ `"New York"`).

Câu hỏi gợi mở:
- "Không có đường đi ra" nghĩa là thành phố đó **không bao giờ đứng ở vị trí đầu** của một cặp. Đúng không?
- Ví dụ 2 cho thấy đường đi cuối cùng **không nhất thiết** nằm cuối mảng. Vậy có thể trả `paths[-1][1]` không?

In [1]:
paths1 = [["London", "New York"], ["New York", "Lima"], ["Lima", "Sao Paulo"]]
paths2 = [["B", "C"], ["D", "B"], ["C", "A"]]
paths3 = [["A", "Z"]]

print("Số đường đi của ví dụ 1:", len(paths1))
print("Đường đi đầu tiên:", paths1[0])
print("Ví dụ 2 — cặp cuối mảng là", paths2[-1], "nhưng đích thật là 'A' (trùng hợp)")
print("Nếu xáo ví dụ 2 thành", [paths2[2], paths2[0], paths2[1]], "thì cặp cuối là", paths2[1])

Số đường đi của ví dụ 1: 3
Đường đi đầu tiên: ['London', 'New York']
Ví dụ 2 — cặp cuối mảng là ['C', 'A'] nhưng đích thật là 'A' (trùng hợp)
Nếu xáo ví dụ 2 thành [['C', 'A'], ['B', 'C'], ['D', 'B']] thì cặp cuối là ['D', 'B']


Quan sát cuối cùng rất quan trọng: nếu ta "đoán" đích là `paths[-1][1]`, ví dụ 2 tình cờ đúng, nhưng chỉ cần đổi thứ tự các cặp là sai (`["D","B"]` ở cuối → đoán `"B"`, sai). Thứ tự các cặp trong mảng **không có ý nghĩa** gì về lộ trình.

## 2. Nền tảng Python cần dùng

### 2.1. List lồng list và chỉ số

`paths` là một **list**, mỗi phần tử lại là một list 2 chuỗi. `paths[i]` là cặp thứ `i`, `paths[i][0]` là thành phố **đi**, `paths[i][1]` là thành phố **đến**.

In [2]:
paths = [["London", "New York"], ["New York", "Lima"]]
print(paths[1])        # ['New York', 'Lima']
print(paths[1][0])     # New York  — điểm đi
print(paths[1][1])     # Lima      — điểm đến
print(paths[-1][1])    # Lima      — chỉ số -1 là phần tử cuối

['New York', 'Lima']
New York
Lima
Lima


### 2.2. Unpacking trong vòng `for`

Khi mỗi phần tử là một cặp, Python cho phép **tách** cặp ra 2 biến ngay trong `for`:

```python
for a, b in paths:   # a = điểm đi, b = điểm đến
```

Đọc là: "với mỗi cặp `[a, b]` trong `paths`". Cách này gọn và dễ đọc hơn `p[0]`, `p[1]`.

In [3]:
paths = [["B", "C"], ["D", "B"], ["C", "A"]]
for a, b in paths:
    print(f"{a} -> {b}")

x, y = ["New York", "Lima"]      # unpacking ngoài vòng for cũng được
print("x =", x, "| y =", y)

B -> C
D -> B
C -> A
x = New York | y = Lima


### 2.3. `set` — tập hợp

`set` là cấu trúc lưu **các phần tử không trùng nhau**, không có thứ tự. Thao tác chính:

| Thao tác | Cú pháp | Ý nghĩa |
|---|---|---|
| Tạo rỗng | `s = set()` | (chú ý: `{}` là **dict** rỗng, không phải set) |
| Tạo từ list | `set(["a", "b", "a"])` | `{'a', 'b'}` — tự bỏ trùng |
| Thêm | `s.add(x)` | |
| Kiểm tra | `x in s` | **trung bình O(1)** nhờ băm (hash) |
| Hiệu | `s1 - s2` | các phần tử có trong `s1` mà không có trong `s2` |

So sánh: `x in list` phải dò từng phần tử → `O(n)`; `x in set` tra bảng băm → trung bình `O(1)`.

In [4]:
sources = set()
sources.add("London")
sources.add("New York")
sources.add("London")              # thêm trùng: không đổi gì
print(sources, "| kích thước:", len(sources))
print("Lima" in sources)           # False
print("London" in sources)         # True
print(type({}), type(set()))       # {} là dict!

{'London', 'New York'} | kích thước: 2
False
True
<class 'dict'> <class 'set'>


In [5]:
A = {"London", "New York", "Lima"}
Bset = {"New York", "Lima", "Sao Paulo"}
print("A - B =", A - Bset)         # {'London'}
print("B - A =", Bset - A)         # {'Sao Paulo'}
diff = Bset - A
print("Lấy phần tử duy nhất bằng pop():", diff.pop())

A - B = {'London'}
B - A = {'Sao Paulo'}
Lấy phần tử duy nhất bằng pop(): Sao Paulo


`set.pop()` lấy ra (và xóa) **một** phần tử bất kỳ. Khi biết chắc tập chỉ có 1 phần tử, `pop()` trả đúng phần tử đó. Một cách khác: `next(iter(s))` lấy phần tử mà không xóa.

### 2.4. `dict` — ánh xạ "đi từ đâu tới đâu"

`dict` lưu cặp **key → value**. Với bài này, `nxt[a] = b` nghĩa là "từ `a` đi tiếp tới `b`". Hàm `nxt.get(key)` trả `None` nếu không có key (thay vì báo lỗi như `nxt[key]`).

In [6]:
nxt = {}
nxt["London"] = "New York"
nxt["New York"] = "Lima"
print(nxt)
print(nxt["London"])            # New York
print(nxt.get("Lima"))          # None — Lima không có đường đi ra
print("Lima" in nxt)            # False: `in` với dict kiểm tra KEY

{'London': 'New York', 'New York': 'Lima'}
New York
None
False


### 2.5. Diễn giải bài toán bằng ngôn ngữ đồ thị

- Mỗi thành phố là một **đỉnh**; mỗi cặp `[A, B]` là một **cạnh có hướng** A → B.
- **Bậc ra** (out-degree) của một đỉnh = số cạnh đi ra từ nó.
- Đích là đỉnh có **bậc ra = 0**.

Vì đề đảm bảo đồ thị là **một đường thẳng**, mọi đỉnh trừ đích đều có bậc ra đúng 1, đích có bậc ra 0. Ta không cần thuật toán đồ thị phức tạp — chỉ cần "ai chưa từng là điểm đi".

In [7]:
from collections import Counter

paths = [["B", "C"], ["D", "B"], ["C", "A"]]
out_degree = Counter(a for a, _ in paths)
cities = {c for pair in paths for c in pair}
for c in sorted(cities):
    print(f"{c}: bậc ra = {out_degree[c]}")

A: bậc ra = 0
B: bậc ra = 1
C: bậc ra = 1
D: bậc ra = 1


Ở cell trên có hai công cụ mới:
- `Counter(...)` (từ `collections`) đếm số lần mỗi giá trị xuất hiện; truy cập key không có trả `0`.
- `{c for pair in paths for c in pair}` là **set comprehension** hai tầng: "với mỗi cặp, với mỗi thành phố trong cặp, đưa vào set". Dấu `_` là tên biến quy ước cho giá trị ta **không dùng**.

## 3. Cách 1 — Brute force: với mỗi điểm đến, dò xem nó có đi ra đâu không

Ý tưởng tự nhiên nhất:
1. Lấy từng thành phố `b` ở vị trí **điểm đến**.
2. Dò lại **toàn bộ** `paths` xem có cặp nào `b` đứng ở vị trí **điểm đi** không.
3. Nếu không có → `b` là đích.

Vì sao chỉ cần xét các điểm đến? Đích chắc chắn là điểm đến của cạnh cuối (đường thẳng có ít nhất 1 cạnh), nên nó nằm trong tập điểm đến.

In [8]:
def dest_city_brute(paths):
    for _, b in paths:                  # b: ứng viên đích
        has_outgoing = False
        for a, _ in paths:              # dò mọi điểm đi
            if a == b:
                has_outgoing = True
                break
        if not has_outgoing:
            return b
    return ""                           # không xảy ra với input hợp lệ

print(dest_city_brute(paths1))   # Sao Paulo
print(dest_city_brute(paths2))   # A
print(dest_city_brute(paths3))   # Z

Sao Paulo
A
Z


### 3.1. Đếm số phép so sánh của brute force

Với `n` cặp, vòng ngoài chạy tối đa `n` lần, vòng trong tối đa `n` lần → tối đa `n²` phép so sánh chuỗi. Với `n = 100` (giới hạn đề) chỉ là 10.000 — vẫn qua, nhưng ta muốn thấy vì sao nó chậm khi `n` lớn.

In [9]:
def count_compares_brute(paths):
    compares = 0
    for _, b in paths:
        found = False
        for a, _ in paths:
            compares += 1
            if a == b:
                found = True
                break
        if not found:
            return b, compares
    return "", compares

# Đường thẳng c0 -> c1 -> ... -> cn, các cặp theo thứ tự tự nhiên:
# ứng viên c(i+1) phải dò tới vị trí i+1 mới thấy, còn đích cn nằm ở cặp cuối
for n in [5, 50, 100]:
    line = [[f"c{i}", f"c{i+1}"] for i in range(n)]
    dest, compares = count_compares_brute(line)
    print(f"{n:>3} cặp → đích {dest}, {compares} phép so sánh (n²/2 ≈ {n*n//2})")

  5 cặp → đích c5, 19 phép so sánh (n²/2 ≈ 12)
 50 cặp → đích c50, 1324 phép so sánh (n²/2 ≈ 1250)
100 cặp → đích c100, 5149 phép so sánh (n²/2 ≈ 5000)


## 4. Nút thắt và insight

**Nút thắt:** với mỗi ứng viên `b`, ta lại **dò lại cả mảng** để hỏi cùng một câu: "`b` có phải điểm đi nào không?". Câu hỏi lặp đi lặp lại trên cùng một dữ liệu.

**Insight:** gom tất cả điểm đi vào **một `set` duy nhất** (làm 1 lần, `O(n)`). Sau đó câu hỏi "`b` có phải điểm đi không?" chỉ còn là `b in sources` — trung bình `O(1)`.

> Quy tắc: **Nếu** bạn đang hỏi "x có xuất hiện trong mảng không?" nhiều lần, **thì** hãy đổ mảng vào `set` một lần.

Cell nhỏ dưới đây tách riêng bước "xây tập điểm đi" để thấy rõ nó chứa gì.

In [10]:
paths = [["B", "C"], ["D", "B"], ["C", "A"]]
sources = {a for a, _ in paths}
print("Tập điểm đi:", sorted(sources))
for _, b in paths:
    print(f"Ứng viên {b!r}: có trong sources? {b in sources}")

Tập điểm đi: ['B', 'C', 'D']
Ứng viên 'C': có trong sources? True
Ứng viên 'B': có trong sources? True
Ứng viên 'A': có trong sources? False


Chỉ có `'A'` không nằm trong tập điểm đi → `'A'` là đích. Đó chính là thuật toán chính.

(`{b!r}` trong f-string in giá trị bằng `repr`, nên chuỗi có dấu nháy — giúp nhìn rõ dấu cách trong `"New York"`.)

## 5. Cách 2 (lời giải chính) — Set các điểm đi

### 5.1. Mổ xẻ từng đoạn

**Đoạn 1 — xây tập điểm đi:**
```python
sources = set()
for a, _ in paths:
    sources.add(a)
```

**Đoạn 2 — tìm điểm đến không thuộc tập:**
```python
for _, b in paths:
    if b not in sources:
        return b
```

Hai vòng `for` **nối tiếp** (không lồng) → tổng `2n` bước, tức `O(n)`.

Vì sao phải xây xong **toàn bộ** `sources` trước rồi mới tìm? Vì cạnh đi ra của một thành phố có thể nằm **sau** nó trong mảng (ví dụ 2: `"B"` là điểm đến ở cặp 0, nhưng điểm đi `"B"` ở cặp 1). Nếu vừa xây vừa kiểm tra, ta sẽ kết luận sai sớm. Mục 13 có code minh họa lỗi này.

In [11]:
def dest_city_set(paths):
    # Đoạn 1: mọi thành phố có đường đi ra
    sources = set()
    for a, _ in paths:
        sources.add(a)
    # Đoạn 2: điểm đến duy nhất không có đường đi ra
    for _, b in paths:
        if b not in sources:
            return b
    return ""

print(dest_city_set(paths1))
print(dest_city_set(paths2))
print(dest_city_set(paths3))

Sao Paulo
A
Z


## 6. TRACE từng bước — ví dụ 2 `[["B","C"],["D","B"],["C","A"]]`

**Pha 1 — xây `sources`:**

| i | cặp `(a, b)` | `sources` trước | `sources` sau |
|---:|---|---|---|
| 0 | (B, C) | {} | {B} |
| 1 | (D, B) | {B} | {B, D} |
| 2 | (C, A) | {B, D} | {B, C, D} |

**Pha 2 — tìm đích:**

| i | `b` | `b in sources`? | Hành động |
|---:|---|---|---|
| 0 | C | True | bỏ qua |
| 1 | B | True | bỏ qua |
| 2 | A | **False** | **return "A"** |

Hàm `trace_dest_city` in đúng các cột này.

In [12]:
def trace_dest_city(paths):
    print("Pha 1 — xây sources")
    sources = set()
    for i, (a, b) in enumerate(paths):
        before = sorted(sources)
        sources.add(a)
        print(f"  i={i} cặp=({a!r}, {b!r}) | trước={before} -> sau={sorted(sources)}")
    print("Pha 2 — tìm điểm đến không thuộc sources")
    for i, (a, b) in enumerate(paths):
        inside = b in sources
        print(f"  i={i} b={b!r} | b in sources = {inside}", "" if inside else "→ TRẢ VỀ")
        if not inside:
            return b
    return ""

ans = trace_dest_city(paths2)
print("Kết quả:", ans)

Pha 1 — xây sources
  i=0 cặp=('B', 'C') | trước=[] -> sau=['B']
  i=1 cặp=('D', 'B') | trước=['B'] -> sau=['B', 'D']
  i=2 cặp=('C', 'A') | trước=['B', 'D'] -> sau=['B', 'C', 'D']
Pha 2 — tìm điểm đến không thuộc sources
  i=0 b='C' | b in sources = True 
  i=1 b='B' | b in sources = True 
  i=2 b='A' | b in sources = False → TRẢ VỀ
Kết quả: A


`enumerate(paths)` sinh cặp `(i, phần_tử)`; vì phần tử lại là cặp `[a, b]` nên ta unpack lồng: `for i, (a, b) in enumerate(paths)` — dấu ngoặc tròn quanh `(a, b)` là bắt buộc.

### 6.1. Trace ví dụ 1 — tên thành phố có dấu cách

In [13]:
ans = trace_dest_city(paths1)
print("Kết quả:", repr(ans))

Pha 1 — xây sources
  i=0 cặp=('London', 'New York') | trước=[] -> sau=['London']
  i=1 cặp=('New York', 'Lima') | trước=['London'] -> sau=['London', 'New York']
  i=2 cặp=('Lima', 'Sao Paulo') | trước=['London', 'New York'] -> sau=['Lima', 'London', 'New York']
Pha 2 — tìm điểm đến không thuộc sources
  i=0 b='New York' | b in sources = True 
  i=1 b='Lima' | b in sources = True 
  i=2 b='Sao Paulo' | b in sources = False → TRẢ VỀ
Kết quả: 'Sao Paulo'


Dấu cách trong `"New York"`, `"Sao Paulo"` không gây vấn đề gì: ta so sánh **cả chuỗi**, không tách từ. Đừng bao giờ `split()` tên thành phố.

## 7. Cách 3 — Đi theo chuỗi (gợi ý của LeetCode)

Hint của đề: *"Bắt đầu ở một thành phố bất kỳ, đi theo đường tới thành phố kế tiếp. Cuối cùng bạn sẽ tới thành phố không có đường đi ra."*

Các bước:
1. Xây `nxt = {a: b for a, b in paths}` — mỗi thành phố có **tối đa một** đường đi ra (vì là đường thẳng).
2. Bắt đầu từ `city = paths[0][0]`.
3. `while city in nxt: city = nxt[city]` — còn đường thì đi tiếp.
4. Khi `city` không còn trong `nxt` → đó là đích.

Vì đồ thị không có vòng nên vòng `while` chắc chắn dừng sau tối đa `n` bước.

In [14]:
def dest_city_walk(paths):
    nxt = {a: b for a, b in paths}     # dict comprehension
    city = paths[0][0]
    while city in nxt:
        city = nxt[city]
    return city

print(dest_city_walk(paths1))
print(dest_city_walk(paths2))
print(dest_city_walk(paths3))

Sao Paulo
A
Z


In [15]:
def trace_walk(paths):
    nxt = {a: b for a, b in paths}
    print("nxt =", nxt)
    city = paths[0][0]
    step = 0
    print(f"  bước {step}: đứng ở {city!r}")
    while city in nxt:
        city = nxt[city]
        step += 1
        print(f"  bước {step}: đi tới {city!r}")
    print(f"  {city!r} không có trong nxt → dừng")
    return city

trace_walk(paths2)

nxt = {'B': 'C', 'D': 'B', 'C': 'A'}
  bước 0: đứng ở 'B'
  bước 1: đi tới 'C'
  bước 2: đi tới 'A'
  'A' không có trong nxt → dừng


'A'

Lưu ý ở ví dụ 2: bắt đầu từ `"B"` (điểm đi của cặp đầu), **không phải** điểm đầu thật `"D"`. Không sao — đi từ bất kỳ đâu trên đường thẳng, cứ tiến về phía trước thì vẫn tới đích.

`{a: b for a, b in paths}` là **dict comprehension**: giống list comprehension nhưng sinh cặp `key: value`.

## 8. Cách 4 — Hiệu tập hợp một dòng

Đích = (tập điểm đến) − (tập điểm đi). Vì chỉ có đúng một phần tử, ta `pop()` nó ra.

In [16]:
def dest_city_diff(paths):
    starts = {a for a, _ in paths}
    ends = {b for _, b in paths}
    return (ends - starts).pop()

for p in [paths1, paths2, paths3]:
    starts = {a for a, _ in p}
    ends = {b for _, b in p}
    print("ends - starts =", ends - starts, "→", dest_city_diff(p))

ends - starts = {'Sao Paulo'} → Sao Paulo
ends - starts = {'A'} → A
ends - starts = {'Z'} → Z


Vì sao `ends - starts` chỉ có 1 phần tử? Trên đường thẳng `c0 → c1 → ... → cn`: điểm đi là `{c0..c(n-1)}`, điểm đến là `{c1..cn}`. Hiệu là `{cn}`. (Ngược lại `starts - ends = {c0}` — điểm xuất phát; dùng ở phần biến thể.)

## 9. Code nộp LeetCode (Python 3)

Bản chính: set các điểm đi. Signature giữ đúng `destCity(self, paths)`.

In [17]:
class Solution:
    def destCity(self, paths: list[list[str]]) -> str:
        sources = {a for a, _ in paths}
        for _, b in paths:
            if b not in sources:
                return b
        return ""

sol = Solution()
print(sol.destCity([["London", "New York"], ["New York", "Lima"], ["Lima", "Sao Paulo"]]))
print(sol.destCity([["B", "C"], ["D", "B"], ["C", "A"]]))
print(sol.destCity([["A", "Z"]]))

Sao Paulo
A
Z


Các bản nộp thay thế (đều Accepted):

```python
class Solution:
    def destCity(self, paths: list[list[str]]) -> str:
        nxt = {a: b for a, b in paths}
        city = paths[0][0]
        while city in nxt:
            city = nxt[city]
        return city
```

```python
class Solution:
    def destCity(self, paths: list[list[str]]) -> str:
        return ({b for _, b in paths} - {a for a, _ in paths}).pop()
```

## 10. Vì sao thuật toán đúng?

**Định nghĩa:** `S` = tập điểm đi. Một thành phố `c` có đường đi ra ⇔ tồn tại cặp `[c, x]` ⇔ `c ∈ S`.

**Invariant pha 1:** sau khi xử lý `i` cặp đầu, `sources` = tập điểm đi của `i` cặp đó. Kết thúc pha 1, `sources = S`.

**Pha 2:**
- Không trả nhầm: hàm trả `b` chỉ khi `b ∉ S`, tức `b` không có đường đi ra → đúng định nghĩa đích.
- Không bỏ sót: đích `d` là điểm đến của cạnh cuối cùng của đường thẳng, nên `d` xuất hiện trong pha 2; và `d ∉ S`. Vì đích duy nhất, không có `b` nào khác thỏa `b ∉ S` → hàm trả đúng `d`.

**Cách 3 (đi theo chuỗi):** mỗi bước `city = nxt[city]` tiến thêm 1 cạnh trên đường thẳng; không có vòng nên không thăm lại đỉnh nào, sau ≤ `n` bước phải tới đỉnh không có trong `nxt` — đó là đích.

## 11. Phân tích Time / Space

Gọi `n = len(paths)`, `L` = độ dài tối đa tên thành phố (≤ 10). Băm/so sánh một chuỗi tốn `O(L)`; vì `L` nhỏ và cố định, ta thường coi là hằng.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n² · L)` | `O(1)` | hai vòng lồng nhau |
| 2. Set điểm đi | `O(n · L)` trung bình | `O(n · L)` | **khuyên dùng** |
| 3. Đi theo chuỗi | `O(n · L)` trung bình | `O(n · L)` | đúng theo hint, cần đồ thị là đường thẳng |
| 4. Hiệu tập hợp | `O(n · L)` trung bình | `O(n · L)` | ngắn nhất, tạo 2 set |

"Trung bình" vì tra cứu set/dict dựa trên băm: trung bình `O(1)`, trường hợp xấu (va chạm băm hiếm) có thể chậm hơn.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual!r}")

sol = Solution()
check("ví dụ 1", sol.destCity(paths1), "Sao Paulo")
check("ví dụ 2", sol.destCity(paths2), "A")
check("ví dụ 3 (1 cặp)", sol.destCity(paths3), "Z")
check("đích nằm ở cặp đầu mảng", sol.destCity([["B", "Z"], ["A", "B"]]), "Z")
check("tên có dấu cách", sol.destCity([["New York", "New Delhi"]]), "New Delhi")
check("phân biệt hoa thường", sol.destCity([["a", "A"]]), "A")
check("phản ví dụ của paths[-1][1]", sol.destCity([["C", "A"], ["B", "C"], ["D", "B"]]), "A")

PASS  ví dụ 1: 'Sao Paulo'
PASS  ví dụ 2: 'A'
PASS  ví dụ 3 (1 cặp): 'Z'
PASS  đích nằm ở cặp đầu mảng: 'Z'
PASS  tên có dấu cách: 'New Delhi'
PASS  phân biệt hoa thường: 'A'
PASS  phản ví dụ của paths[-1][1]: 'A'


### 12.1. Đối chiếu 4 cách trên đường thẳng ngẫu nhiên (có seed)

Ta tự sinh đường thẳng ngẫu nhiên: chọn một hoán vị các tên, nối liên tiếp, rồi **xáo trộn** thứ tự các cặp. Đáp án đúng biết trước là tên cuối cùng của hoán vị.

In [19]:
import random

def random_line(rng, n):
    names = [f"city{k}" for k in range(n + 1)]
    rng.shuffle(names)
    pairs = [[names[i], names[i + 1]] for i in range(n)]
    rng.shuffle(pairs)
    return pairs, names[-1]

rng = random.Random(1436)
methods = [dest_city_brute, dest_city_set, dest_city_walk, dest_city_diff]
for _ in range(300):
    n = rng.randint(1, 100)
    p, expected = random_line(rng, n)
    for f in methods:
        assert f(p) == expected, (f.__name__, p)
print("300 đường thẳng ngẫu nhiên: 4 cách đều khớp đáp án")

300 đường thẳng ngẫu nhiên: 4 cách đều khớp đáp án


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Trả về điểm đến của cặp cuối mảng

```python
return paths[-1][1]
```
**Vì sao sai:** thứ tự các cặp trong mảng là tùy ý, không phải thứ tự lộ trình.

In [20]:
def wrong_last_pair(paths):
    return paths[-1][1]

p = [["C", "A"], ["B", "C"], ["D", "B"]]
print("Sai:", wrong_last_pair(p), "| Đúng:", dest_city_set(p))

Sai: B | Đúng: A


### Lỗi 2 — Vừa xây set vừa kiểm tra (kết luận quá sớm)

```python
sources = set()
for a, b in paths:
    sources.add(a)
    if b not in sources:   # cạnh đi ra của b có thể nằm SAU!
        return b
```
**Vì sao sai:** lúc xét `b`, có thể ta chưa gặp cặp mà `b` là điểm đi. **Sửa:** xây xong toàn bộ `sources` rồi mới kiểm tra (2 vòng tách rời).

In [21]:
def wrong_one_pass(paths):
    sources = set()
    for a, b in paths:
        sources.add(a)
        if b not in sources:
            return b
    return ""

p = [["B", "C"], ["D", "B"], ["C", "A"]]
print("Sai:", wrong_one_pass(p), "| Đúng:", dest_city_set(p))

Sai: C | Đúng: A


### Lỗi 3 — Nhầm vai trò: tìm điểm đi không phải điểm đến

```python
ends = {b for _, b in paths}
for a, _ in paths:
    if a not in ends:
        return a           # đây là điểm XUẤT PHÁT, không phải đích
```
**Sửa:** đích là thứ **không bao giờ đi ra** → kiểm tra điểm **đến** không thuộc tập điểm **đi**.

In [22]:
def wrong_role(paths):
    ends = {b for _, b in paths}
    for a, _ in paths:
        if a not in ends:
            return a
    return ""

print("Sai:", wrong_role(paths1), "| Đúng:", dest_city_set(paths1))

Sai: London | Đúng: Sao Paulo


### Lỗi 4 — Dùng `{}` để tạo set rỗng

```python
sources = {}
sources.add("A")   # AttributeError: 'dict' object has no attribute 'add'
```
**Sửa:** `set()`. Cell dưới bắt lỗi bằng `try/except` để thấy thông báo mà không làm dừng notebook.

In [23]:
try:
    s = {}
    s.add("A")
except AttributeError as e:
    print("Lỗi:", e)
s = set()
s.add("A")
print("Đúng:", s)

Lỗi: 'dict' object has no attribute 'add'
Đúng: {'A'}


### Lỗi 5 — Dùng list thay set cho `sources`

Code vẫn đúng nhưng `b not in sources_list` là `O(n)` → tổng `O(n²)`, mất ý nghĩa tối ưu. Với `n` lớn sẽ thấy chênh lệch rõ:

In [24]:
import time

def dest_city_list(paths):
    sources = [a for a, _ in paths]
    for _, b in paths:
        if b not in sources:
            return b

line = [[f"c{i}", f"c{i+1}"] for i in range(4000)]
t0 = time.perf_counter(); r1 = dest_city_list(line); t1 = time.perf_counter()
r2 = dest_city_set(line); t2 = time.perf_counter()
print("Cùng kết quả:", r1 == r2, "| bản set nhanh hơn bản list:", (t2 - t1) < (t1 - t0))

Cùng kết quả: True | bản set nhanh hơn bản list: True


(Thời gian cụ thể thay đổi theo máy; với 4000 cặp, bản `list` phải làm khoảng 8 triệu phép so sánh, bản `set` chỉ khoảng 8000 lần tra băm.)

## 14. Mẹo nhận dạng pattern

- Đề cho **danh sách cạnh** `[u, v]` và hỏi đỉnh "không đi ra" / "không có ai trỏ tới" → nghĩ tới **bậc vào / bậc ra** và **set**.
- Câu hỏi dạng "phần tử nào **có ở A mà không có ở B**" → **hiệu tập hợp** `A - B`.
- Đề đảm bảo "đường thẳng, không vòng" → có thể **đi theo chuỗi** bằng `dict`.
- Bài cùng họ: *Find the Town Judge* (997), *Find Center of Star Graph* (1791), *Find All Numbers Disappeared in an Array* (448, hiệu tập hợp).

## 15. Biến thể

### 15.1. Tìm thành phố xuất phát
Đối xứng hoàn toàn: điểm đi không bao giờ là điểm đến.

In [25]:
def start_city(paths):
    ends = {b for _, b in paths}
    for a, _ in paths:
        if a not in ends:
            return a
    return ""

print(start_city(paths1))   # London
print(start_city(paths2))   # D

London
D


### 15.2. In toàn bộ lộ trình theo đúng thứ tự
Kết hợp 15.1 (tìm điểm đầu) với cách 3 (đi theo `dict`).

In [26]:
def full_route(paths):
    nxt = {a: b for a, b in paths}
    city = start_city(paths)
    route = [city]
    while city in nxt:
        city = nxt[city]
        route.append(city)
    return route

print(" -> ".join(full_route(paths1)))
print(" -> ".join(full_route(paths2)))

London -> New York -> Lima -> Sao Paulo
D -> B -> C -> A


### 15.3. Nhiều đường rời nhau — trả mọi thành phố đích
Nếu bỏ đảm bảo "một đường thẳng", có thể có nhiều đích; hiệu tập hợp vẫn dùng được, chỉ cần trả cả tập (sắp xếp để kết quả ổn định).

In [27]:
def all_dest_cities(paths):
    return sorted({b for _, b in paths} - {a for a, _ in paths})

print(all_dest_cities([["A", "B"], ["B", "C"], ["X", "Y"]]))   # ['C', 'Y']

['C', 'Y']


## 16. Bài tập luyện tập (tăng dần)

**Bài 1 — Tính tay.** Với `paths = [["Q","R"],["P","Q"],["R","S"],["O","P"]]`: viết tập điểm đi, tập điểm đến, hiệu hai tập. Đích là gì? Điểm xuất phát là gì?

**Bài 2 — Dự đoán output.** Không chạy, đoán:
```python
s = {"x", "y"}; t = {"y", "z"}
print(s - t, t - s, len(s | t))
```

Các bài 3–6 có cell TODO bên dưới.

### Bài 3 — TODO: tự viết lại lời giải set

In [28]:
def exercise_dest_city(paths):
    # TODO 1: tạo set các điểm đi
    # TODO 2: duyệt điểm đến, trả về điểm không thuộc set
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_dest_city([["B","C"],["D","B"],["C","A"]]) == "A"

### Bài 4 — TODO: đếm số thành phố khác nhau
Trả về số thành phố **khác nhau** xuất hiện trong `paths`.

In [29]:
def exercise_count_cities(paths):
    # TODO: gom mọi tên vào một set rồi lấy len
    pass

# assert exercise_count_cities([["B","C"],["D","B"],["C","A"]]) == 4

### Bài 5 — TODO: kiểm tra `paths` có thật sự là một đường thẳng
Trả `True` nếu: mỗi thành phố có bậc ra ≤ 1 và bậc vào ≤ 1, có đúng 1 điểm xuất phát, và đi từ điểm xuất phát thăm hết mọi thành phố.

In [30]:
def exercise_is_line(paths):
    # TODO 1: xây nxt, kiểm tra không có điểm đi trùng
    # TODO 2: kiểm tra không có điểm đến trùng
    # TODO 3: tìm điểm xuất phát duy nhất, đi theo nxt, đếm số đỉnh thăm được
    pass

# assert exercise_is_line([["B","C"],["D","B"],["C","A"]]) == True
# assert exercise_is_line([["A","B"],["B","A"]]) == False      # vòng
# assert exercise_is_line([["A","B"],["A","C"]]) == False      # A có 2 đường ra

### Bài 6 — Mini challenge: thành phố cách đích `k` bước
Cho đường thẳng hợp lệ và số `k ≥ 0`. Trả về thành phố mà đi thêm đúng `k` bước thì tới đích (`k = 0` → chính là đích). Nếu không tồn tại trả `""`.

In [31]:
def exercise_k_before_dest(paths, k):
    # TODO: gợi ý — xây dict NGƯỢC prev[b] = a, đi lùi từ đích k bước
    pass

# assert exercise_k_before_dest([["London","New York"],["New York","Lima"],["Lima","Sao Paulo"]], 2) == "New York"
# assert exercise_k_before_dest([["A","Z"]], 5) == ""

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Đích không bao giờ là điểm đi. | `set` + 2 vòng tách rời. | `src = {a for a,_ in paths}`; `for _, b in paths: if b not in src: return b` |
| 4 | Thành phố có thể lặp lại ở 2 cặp. | set comprehension 2 tầng. | `len({c for p in paths for c in p})` |
| 5 | Đường thẳng ⇔ bậc ≤ 1, một đầu, liên thông. | `dict` cho `nxt`, `set` cho điểm đến. | nếu `len(nxt) != len(paths)` → có điểm đi trùng; đi từ start đếm `len(route) == số thành phố` |
| 6 | Đi lùi dễ hơn đi tiến. | `prev = {b: a for a, b in paths}` | `city = đích`; lặp `k` lần: nếu `city not in prev` → `""`; `city = prev[city]` |

## 18. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** điểm đi `{Q, P, R, O}`, điểm đến `{R, Q, S, P}`; đến − đi = `{S}` → đích `S`; đi − đến = `{O}` → xuất phát `O`.
**Bài 2:** `{'x'} {'z'} 3`.

In [32]:
def answer_dest_city(paths):
    src = {a for a, _ in paths}
    for _, b in paths:
        if b not in src:
            return b
    return ""

def answer_count_cities(paths):
    return len({c for pair in paths for c in pair})

def answer_is_line(paths):
    nxt = {}
    for a, b in paths:
        if a in nxt:
            return False                 # a có 2 đường ra
        nxt[a] = b
    ends = [b for _, b in paths]
    if len(set(ends)) != len(ends):
        return False                     # có thành phố 2 đường vào
    starts = [a for a, _ in paths if a not in set(ends)]
    if len(starts) != 1:
        return False                     # vòng (0 điểm đầu) hoặc nhiều đường rời
    city, seen = starts[0], 1
    while city in nxt:
        city = nxt[city]
        seen += 1
    return seen == answer_count_cities(paths)

def answer_k_before_dest(paths, k):
    prev = {b: a for a, b in paths}
    city = answer_dest_city(paths)
    for _ in range(k):
        if city not in prev:
            return ""
        city = prev[city]
    return city

assert answer_dest_city([["B","C"],["D","B"],["C","A"]]) == "A"
assert answer_count_cities([["B","C"],["D","B"],["C","A"]]) == 4
assert answer_is_line([["B","C"],["D","B"],["C","A"]]) == True
assert answer_is_line([["A","B"],["B","A"]]) == False
assert answer_is_line([["A","B"],["A","C"]]) == False
assert answer_is_line([["A","B"],["C","D"]]) == False
assert answer_k_before_dest([["London","New York"],["New York","Lima"],["Lima","Sao Paulo"]], 2) == "New York"
assert answer_k_before_dest([["London","New York"],["New York","Lima"],["Lima","Sao Paulo"]], 0) == "Sao Paulo"
assert answer_k_before_dest([["A","Z"]], 5) == ""
print("Đáp án bài 3–6: PASS toàn bộ test")

Đáp án bài 3–6: PASS toàn bộ test


In [33]:
# Kiểm tra lại bài 1 và bài 2 bằng code
p = [["Q","R"],["P","Q"],["R","S"],["O","P"]]
print("đích:", answer_dest_city(p), "| xuất phát:", start_city(p))
s = {"x", "y"}; t = {"y", "z"}
print(s - t, t - s, len(s | t))

đích: S | xuất phát: O
{'x'} {'z'} 3


## 19. Template nhớ nhanh + checklist submit

```python
# "Phần tử nào ở vai trò X nhưng KHÔNG BAO GIỜ ở vai trò Y?"
seen_Y = {y for ..., y in pairs}       # gom vai trò Y vào set, 1 lần
for x in candidates_X:
    if x not in seen_Y:                 # tra O(1) trung bình
        return x
```

**Checklist trước khi nộp**
- [ ] Đích là điểm **đến** không thuộc tập điểm **đi** (không nhầm chiều).
- [ ] Xây **xong** set rồi mới kiểm tra — không làm trong cùng một vòng.
- [ ] Không dựa vào thứ tự các cặp trong mảng (`paths[-1][1]` là sai).
- [ ] So sánh nguyên chuỗi, giữ dấu cách và hoa/thường.
- [ ] Tạo set rỗng bằng `set()`, không phải `{}`.
- [ ] Method tên `destCity`, trả về **chuỗi**.